# Romberg balance score — validation

This notebook reproduces the validation analyses of the Romberg balance score reported in the paper.
It starts from the **test set with the score already computed** (one row per Romberg test, eyes open `.0` / eyes closed `.1`).
Feature extraction and model training are not part of this repository.

**Contents**

1. Setup — imports and configuration
2. Functions — plotting, performance metrics, reliability, PDF report and longitudinal analysis
3. Data loading and sample description
4. Cross-sectional performance (ROC, boxplots, test-retest) — PDF report
5. Score by clinical classification
6. Correlation with clinical scales
7. Longitudinal analysis anchored on R-ODS
8. Score change vs MICARS-SARA (Figure 5)

**Input files** (in `DATA_DIR`)

| File | Content |
|---|---|
| `test_data_romberg_retrained_scores.csv` | One row per test: `SCORE`, identifiers, pathology, clinical classifications, `ELAPSED_DAYS`, `RETEST`, `QUESTIONNAIRE_R-ODS_centile` |
| `smwt_db_mg.scales_quest.csv` | Clinical scales and questionnaires per session (`ID_SESSION`) |

All figures and tables are written to `OUTPUT_DIR`.

## 1. Setup

In [ ]:
import io
import os
import textwrap
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sns
import statsmodels.formula.api as smf
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import (
    BaseDocTemplate,
    Frame,
    Image,
    PageBreak,
    PageTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
)
from scipy import stats
from scipy.stats import gaussian_kde
from sklearn import metrics
from statannotations.Annotator import Annotator
from tableone import TableOne
from datetime import datetime

In [ ]:
# Paths
DATA_DIR = ""
OUTPUT_DIR = ""
if OUTPUT_DIR:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

SCORES_FILE = os.path.join(DATA_DIR, "test_data_with_scores.csv")
SCALES_FILE = os.path.join(DATA_DIR, "scales_quest.csv")

# Analysis parameters
SCORE_FIELD = "SCORE"
TIMELAPSE_RETEST = 150          # max. days between test and retest (reliability)
SCALES_TO_COMPARE = [
    "SCALE_MICARS-SARA_total",
    "SCALE_MICARS-SARA_1",
    "SCALE_MICARS-SARA_4",
    "SCALE_MICARS-SARA_5",
]

## 2. Functions

### 2.1 Plotting utilities
Box/strip plots with Mann-Whitney annotations between groups, ROC curves and scatter plots of the score against clinical variables.

In [ ]:
DEFAULT_PALETTE = [
    "gray",
    "#d9ed92",
    "#b5e48c",
    "#99d98c",
    "#76c893",
    "#52b69a",
    "#34a0a4",
    "#168aad",
    "#1a759f",
    "#1e6091",
    "#184e77",
]


def select_divergent_colors(color_array, num_colors):
    num_colors = min(num_colors, len(color_array))
    selected_colors = [
        color_array[i]
        for i in np.linspace(0, len(color_array) - 1, num_colors, dtype=int)
    ]
    return selected_colors


def draw_roc(y_true, predictor, label, axes, color):
    fpr, tpr, thresholds = metrics.roc_curve(y_true, predictor)
    axes.plot(fpr, tpr, label=label, color=color, linewidth=1.5)
    plt.ylabel("True Positive Rate")
    plt.xlabel("False Positive Rate")
    plt.legend(loc=4)


def plot_roc_score(nested_ys, nested_scores, labels, color_palette=DEFAULT_PALETTE):
    fig, ax = plt.subplots(1, 1, figsize=(3.4, 3), layout="tight")
    ephpalette = sns.color_palette(
        select_divergent_colors(color_palette, len(nested_ys))
    )
    for i, y in enumerate(nested_ys):
        draw_roc(y, nested_scores[i], labels[i], ax, ephpalette[i])
    return fig


def draw_box(plotdata, group, feature, color_palette, axes):

    if isinstance(plotdata[group].dtype, pd.CategoricalDtype):
        plotdata[group] = plotdata[group].cat.remove_unused_categories()

    PROPS = {
        "boxprops": {"facecolor": "none", "edgecolor": "black"},
        "medianprops": {"color": "black", "linewidth": 3},
        "whiskerprops": {"color": "black"},
        "capprops": {"color": "black"},
    }

    p = sns.stripplot(
        x=group,
        y=feature,
        data=plotdata,
        hue=group,
        palette=color_palette,
        alpha=0.6,
        size=7,
        marker="o",
        zorder=2,
        ax=axes,
    )
    p = sns.boxplot(
        x=group,
        y=feature,
        data=plotdata,
        showfliers=False,
        color="white",
        **PROPS,
        zorder=3,
        ax=axes,
    )
    p.set(xlabel=group, ylabel=feature)
    if axes.get_legend() is not None:
        axes.get_legend().remove()

    try:
        pairs = list(combinations(plotdata[group].dropna().unique(), 2))
        annotator = Annotator(p, pairs, data=plotdata, x=group, y=feature)
        annotator.configure(
            text_format="star",
            loc="inside",
            test="Mann-Whitney",
            text_offset=-2,
            line_height=0.02,
            hide_non_significant=True,
            verbose=0,
        )
        annotator.apply_and_annotate()
    except ValueError as e:
        # If there are not enough data points to perform the test, we skip the annotation
        print(f"Skipping annotation for {group} due to insufficient data: {e}")


def bplot_score(plot_data, groups_list, scores, filename, color_palette=DEFAULT_PALETTE):

    valid_order = ["Control", "Normal", "Mild", "Moderate", "Severe", "Falling"]

    width = 3.5 if len(groups_list) == 1 else 8
    fig, ax = plt.subplots(
        1,
        len(groups_list),
        figsize=(width, 4),
        sharey=True,
        layout="tight",
    )
    for i, group in enumerate(groups_list):

        # --- Order categories ---
        if 'SEVERITY' in group or 'OPEN_EYES_BALANCING' in group or 'ATAXIA' in group or 'CLOSE_EYES_BALANCING' in group:
            plot_data[group] = pd.Categorical(
                plot_data[group],
                categories=valid_order,
                ordered=True
            )

        axes = ax[i] if len(groups_list) > 1 else ax
        if group == 'PAT':
            group_values = sorted(plot_data[group].dropna().unique())
            colors_ = list(select_divergent_colors(color_palette, len(group_values)))
            if not colors_ or any(len(c) < 2 for c in colors_):
                colors_ = sns.color_palette("Set2", len(group_values))
            ephpalette = sns.color_palette(colors_)
        elif group == 'PATHOLOGY':
            ephpalette = ['grey', '#52b69a']
        else:
            group_values = [g for g in valid_order if g in plot_data[group].unique()]
            colors_ = list(select_divergent_colors(color_palette, len(group_values)))
            if not colors_ or any(len(c) < 2 for c in colors_):
                colors_ = sns.color_palette("Set2", len(group_values))
            ephpalette = sns.color_palette(colors_)

        draw_box(plot_data, group, scores, ephpalette, axes)

        if i == 0:
            axes.set(xlabel=group, ylabel=scores)
        else:
            axes.set(xlabel=group, ylabel=None)

    if filename is not None:
        fig.savefig(filename, dpi=300, bbox_inches="tight")

    return fig


def draw_scatter(plotdata, x_var, y_var, group, color_palette, axes):

    group_values = list(plotdata[group].unique())
    marker_dict = {key: ("X" if key == "Control" else "o") for key in group_values}

    plotdata[x_var] = np.float64(plotdata[x_var])
    plotdata[y_var] = np.float64(plotdata[y_var])

    p = sns.scatterplot(
        data=plotdata,
        hue=group,
        palette=color_palette,
        style=group,
        markers=marker_dict,
        x=x_var,
        y=y_var,
        s=60,
        ax=axes,
    )
    r_value, p_value = stats.spearmanr(
        plotdata[x_var], plotdata[y_var], nan_policy="omit"
    )
    axes.set_title(f"rho = {r_value:.2f}; p = {p_value:.5f}", fontsize=10)
    p.set(xlabel=x_var, ylabel=y_var)
    if axes.get_legend() is not None:
        axes.get_legend().remove()


def scatter_with_gaussian_kde(ax, x, y):
    # https://stackoverflow.com/a/20107592/3015186 (Joel Kington)
    idx = ~np.isnan(x + y)
    xy = np.vstack([x[idx], y[idx]])
    z = gaussian_kde(xy)(xy)
    ax.scatter(x[idx], y[idx], c=z, s=30)


def scatter_score(
    plotdata, group, y_var, list_x_vars, color_palette=DEFAULT_PALETTE, mode="scatter"
):
    width = 3.5 if len(list_x_vars) == 1 else 8
    fig, ax = plt.subplots(
        1,
        len(list_x_vars),
        figsize=(width, 3),
        sharey=True,
        layout="tight",
    )
    group_values = list(plotdata[group].unique())
    ephpalette = sns.color_palette(
        select_divergent_colors(color_palette, len(group_values))
    )
    for i, x_var in enumerate(list_x_vars):
        axes = ax[i] if len(list_x_vars) > 1 else ax
        if mode == "scatter":
            draw_scatter(plotdata, x_var, y_var, group, ephpalette, axes)
        elif mode == "density":
            scatter_with_gaussian_kde(axes, plotdata[x_var], plotdata[y_var])
        sns.regplot(
            data=plotdata[plotdata[group] != "Control"],
            x=x_var,
            y=y_var,
            scatter=False,
            robust=True,
            color="black",
            ax=axes,
        )
        x_label_plot = x_var
        if x_var == "SCALE_MICARS-SARA_4":
            x_label_plot = "SCALE_MICARS-SARA_5"
        elif x_var == "SCALE_MICARS-SARA_5":
            x_label_plot = "SCALE_MICARS-SARA_6"
        x_label_plot = (
            x_label_plot.replace("QUESTIONNAIRE_", "")
            .replace("SCALE_", "")
            .replace("OVERVIEW_", "")
        )
        axes.set(xlabel=x_label_plot, ylabel=y_var if i == 0 else None)

    return fig



def bland_altman_plot(data, feature, main_group, ephpalette, axes):
    data1 = np.asarray(data[feature][0])
    data2 = np.asarray(data[feature][1])
    mean = np.mean([data1, data2], axis=0)
    diff = data1 - data2  # Difference between data1 and data2
    md = np.mean(diff)  # Mean of the difference
    sd = np.std(diff, axis=0)  # Standard deviation of the difference

    sns.scatterplot(
        data=data,
        x=mean,
        y=diff,
        s=60,
        hue=main_group,
        palette=ephpalette,
        ax=axes,
    )
    axes.axhline(md, color="black", alpha=0.7, linewidth=1.4, linestyle="dotted")
    axes.axhline(md + 1.96 * sd, color="gray", alpha=0.7, linestyle="--")
    axes.axhline(md - 1.96 * sd, color="gray", alpha=0.7, linestyle="--")

def draw_retest_violin(plotdata, feature, retest, subject, color, axes):

    sns.lineplot(
        data=plotdata,
        y=feature,
        color=color,
        x=plotdata[retest],
        units=subject,
        estimator=None,
        ax=axes,
        lw=1.2,
        alpha=0.5,
    )
    sns.scatterplot(
        data=plotdata,
        y=feature,
        s=60,
        alpha=0.9,
        color=color,
        x=plotdata[retest],
        ax=axes,
    )
    # axes.legend_.remove()
    axes.set_xticks([0, 1], labels=["Test", "Retest"])
    axes.set_xlabel(None)
    
def add_vlines(plotdata, feature, axes):
    # draw vertical lines for test-retest
    axes.vlines(
        x=0,
        ymin=plotdata[feature].min(),
        ymax=plotdata[feature].max(),
        color="black",
        alpha=0.7,
        linewidth=1.4,
        linestyles="dotted",
    )
    axes.vlines(
        x=1,
        ymin=plotdata[feature].min(),
        ymax=plotdata[feature].max(),
        color="black",
        alpha=0.7,
        linewidth=1.4,
        linestyles="dotted",
    )

### 2.2 Cross-sectional performance
ROC analysis for three comparisons: *Control vs Pathology*, *Control vs Mild* and *Mild vs Moderate/Severe*.
For each one, AUC and the sensitivity/specificity/accuracy at the Youden-optimal threshold are reported.

In [ ]:
def get_roc_metrics(ys, scores, labels=None):
    """
    Function that, using a list of y (0 and 1) and a list of scores, extract roc metrics
    """
    fpr = []
    tpr = []
    roc_auc = []
    youden = []
    accuracy = []
    for i, score in enumerate(scores):
        # compute FPR and TPR from y test set and predicted probabilities
        fpr_tmp, tpr_tmp, thresholds = metrics.roc_curve(ys[i], score)
        # compute ROC AUC
        roc_auc_tmp = round(metrics.auc(fpr_tmp, tpr_tmp), 4)
        # youden's index
        idx = np.argmax(tpr_tmp - fpr_tmp)
        youden_tmp = round(thresholds[idx], 2)
        fpr_youden_tmp = round(fpr_tmp[idx], 4)
        tpr_youden_tmp = round(tpr_tmp[idx], 4)
        # accuracy for that specific value
        accuracy_tmp = round(
            metrics.accuracy_score(ys[i], np.where(score >= youden_tmp, 1, 0)), 4
        )
        fpr.append(fpr_youden_tmp)
        tpr.append(tpr_youden_tmp)
        roc_auc.append(roc_auc_tmp)
        youden.append(youden_tmp)
        accuracy.append(accuracy_tmp)

    return fpr, tpr, accuracy, roc_auc, youden


def evaluate_model_performance(data, severity_fields, score="SCORES"):
    roc_fig_nested = []
    table_performance_nested = []
    for severity in severity_fields:
        y_label = data["PATHOLOGY"].replace({"Control": 1, "Pathology": 0}).values
        scores = data[score].values
        y_label_low = np.where(
            data["PATHOLOGY"] == "Control",
            1,
            np.where(data[severity] == "Mild", 0, np.nan),
        )
        scores_low = scores[~np.isnan(y_label_low)]
        y_label_low = y_label_low[~np.isnan(y_label_low)]

        severe_group = ["Moderate", "Severe"]
        y_label_lowhigh = np.where(
            data[severity] == "Mild", 1, np.where(data[severity].isin(severe_group), 0, np.nan)
        )
        scores_lowhigh = scores[~np.isnan(y_label_lowhigh)]
        y_label_lowhigh = y_label_lowhigh[~np.isnan(y_label_lowhigh)]

        nested_ys = [y_label, y_label_low, y_label_lowhigh]
        nested_scores = [scores, scores_low, scores_lowhigh]

        labels = ["Control vs Pathology", "Control vs Mild", "Mild vs Severe"]
        fpr, tpr, accuracy, roc_auc, youden = get_roc_metrics(nested_ys, nested_scores)
        roc_fig = plot_roc_score(nested_ys, nested_scores, labels)
        roc_fig_nested.append(roc_fig)

        table_performance = pd.DataFrame(
            {
                "Comparison": labels,
                "AUC": roc_auc,
                "FPR (1-Specificity)": fpr,
                "TPR (Sensitivity)": tpr,
                "Accuracy": accuracy,
            }
        )
        table_performance_nested.append(table_performance)
    return roc_fig_nested, table_performance_nested

### 2.3 Test-retest reliability
First and last test of each patient within `TIMELAPSE_RETEST` days: paired t-test, Cronbach's alpha, Pearson r, ICC and repeatability coefficient.

In [ ]:
def get_trt_data(data):
    """
    Function extract two dataframes: test and retest. For validation purposes
    """
    if "ID_PATIENT" in data.columns:
        key = "ID_PATIENT"
    else:
        key = "SUBJECT"

    # Count follow-ups per patient and filter out patients with only a single follow-up
    followup_counts = data.groupby(key)["RETEST"].transform("count")
    data_multiple_followups = data[followup_counts > 1]

    # Sort by ID_PATIENT and RETEST to ensure the order of follow-ups
    data_sorted = data_multiple_followups.sort_values(by=[key, "RETEST"])

    # Group by patient and extract the first and last rows for each patient
    first_followup = data_sorted.groupby(key).first().reset_index()
    last_followup = data_sorted.groupby(key).last().reset_index()

    # Get only those retest with more of one day of difference
    retest_tmp = last_followup[last_followup["ELAPSED_DAYS"] > 0]
    ids_retest = retest_tmp["ID_PATIENT"].values

    # Drop duplicates to ensure unique entries for each ID_PATIENT
    test = first_followup[
        first_followup["ID_PATIENT"].isin(ids_retest)
    ].drop_duplicates(subset=[key])
    retest = last_followup[
        last_followup["ID_PATIENT"].isin(ids_retest)
    ].drop_duplicates(subset=[key])
    test["RETEST"] = 0
    retest["RETEST"] = 1

    return test, retest


def robustness_validation(df, df1, var):
    """
    Function to validate score results using pvalue in order to separate test-retest, cronbach alpha and ICC in order to evaluate reliability
    between test-retest and correlation between test-retest
    """
    group_0_scores = df[var].tolist()
    group_1_scores = df1[var].tolist()
    data = pd.DataFrame({"Group 0": group_0_scores, "Group 1": group_1_scores})
    # Paired t-test p-value
    t_statistic_01, p_value_01 = stats.ttest_rel(group_0_scores, group_1_scores)
    # Cronbach's alpha between test and retest
    alpha_between_groups = pg.cronbach_alpha(data)
    # Pearson correlation between test and retest scores
    correlation = np.corrcoef(group_0_scores, group_1_scores)[0, 1]
    # ICC
    data["id"] = range(0, len(data), 1)
    data_melt = pd.melt(data, id_vars="id")
    try:
        icc = pg.intraclass_corr(
            data=data_melt,
            targets="id",
            ratings="value",
            raters="variable",
            nan_policy="omit",
        )
        icc_0 = icc["ICC"][0]
    except:
        icc_0 = np.nan
    # Repeatability coefficient
    # https://rowannicholls.github.io/python/statistics/agreement/repeatability_coefficient.html
    var_list = []
    for i, score0 in enumerate(group_0_scores):
        var_tmp = np.var([score0, group_1_scores[i]], ddof=1)
        var_list.append(var_tmp)
    var_w = np.mean(var_list)
    s_w = np.sqrt(var_w)
    rc = 1.96 * np.sqrt(2) * s_w

    # Number of tests used
    n = data["Group 0"] * data["Group 1"]

    return pd.DataFrame(
        {
            "Paired_ttest": p_value_01,
            "Cronbach_alpha": alpha_between_groups[0],
            "Pearson R": correlation,
            "ICC": icc_0,
            "Repeatability Coef": rc,
            "Subjects with retest": n.count(),
        },
        index=[0],
    )
    


def test_retest_plot(
    t_rt_data,
    scores,
    main_group,
    info_Subject,
    info_retest,
    color_palette=DEFAULT_PALETTE,
):
    ## correlation score t vs rt
    fig, ax = plt.subplots(2, 2, figsize=(6.5, 6), layout="tight")
    t_rt_data[main_group] = pd.Categorical(t_rt_data[main_group])
    group_values = list(t_rt_data[main_group].unique())
    ephpalette = select_divergent_colors(color_palette, len(group_values))

    # plot violins control
    draw_retest_violin(
        t_rt_data[t_rt_data[main_group] == "Control"],
        scores,
        info_retest,
        info_Subject,
        ephpalette[0],
        ax[0, 0],
    )
    add_vlines(t_rt_data, scores, ax[0, 0])
    ax[0, 0].title.set_text("Test-retest in controls")

    # plot violins path
    try:
        draw_retest_violin(
            t_rt_data[t_rt_data[main_group] != "Control"],
            scores,
            info_retest,
            info_Subject,
            ephpalette[1],
            ax[0, 1],
        )
        add_vlines(t_rt_data, scores, ax[0, 1])
    except:
        pass
    ax[0, 1].title.set_text("Test-retest in pathology")
    ax[0, 1].set_ylabel(None)

    ## create cross data needed for the scatter test-retest
    t_rt_data_pivot = t_rt_data.pivot_table(
        index=[info_Subject, main_group], columns=info_retest, values=[scores]
    )
    sns.scatterplot(
        data=t_rt_data_pivot,
        x=t_rt_data_pivot[scores][0],
        y=t_rt_data_pivot[scores][1],
        s=60,
        hue=main_group,
        palette=sns.color_palette(ephpalette),
        ax=ax[1, 0],
    )
    ax[1, 0].plot(
        [
            np.min([t_rt_data_pivot[scores][0], t_rt_data_pivot[scores][1]]),
            np.max([t_rt_data_pivot[scores][0], t_rt_data_pivot[scores][1]]),
        ],
        [
            np.min([t_rt_data_pivot[scores][0], t_rt_data_pivot[scores][1]]),
            np.max([t_rt_data_pivot[scores][0], t_rt_data_pivot[scores][1]]),
        ],
        color="black",
        alpha=0.7,
        linewidth=1.4,
        linestyle="dotted",
    )
    ax[1, 0].title.set_text("Correlation score test-retest")
    ax[1, 0].set_xlabel("%s at baseline" % scores)
    ax[1, 0].set_ylabel("%s at retest" % scores)
    if ax[1, 0].get_legend() is not None:
        ax[1, 0].get_legend().remove()

    ## - bland-altmann plot
    bland_altman_plot(
        t_rt_data_pivot,
        scores,
        main_group,
        ephpalette,
        ax[1, 1],
    )
    ax[1, 1].title.set_text("Bland-Altman plot")
    ax[1, 1].set_xlabel("Mean %s" % scores)
    ax[1, 1].set_ylabel("%s basal-retest difference" % scores)

    if ax[1, 1].get_legend() is not None:
        ax[1, 1].get_legend().remove()

    return fig



### 2.4 PDF report
Builds a PDF with demographics, features used, ROC curves and performance tables, boxplots, scatter plots and test-retest metrics.

In [ ]:
FEATURE_FIELDS = [
    "INSOLE_VFO-NORM",
    "INSOLE_COP_X",
    "INSOLE_COP_Y",
    "TRUNK_ANGLE_X",
    "TRUNK_ANGLE_Y",
    "TRUNK_ANGLE_Z",
]

# Raw signals excluded from the score (same as in training) - because we used calculated features
FEATURES_TO_REMOVE = [
    "INSOLE_COP_X_L", "INSOLE_COP_Y_L", "INSOLE_COP_X_R", "INSOLE_COP_Y_R",
    "TRUNK_ACC_X", "TRUNK_ACC_Y", "TRUNK_ACC_Z", "INSOLE_VFO-NORM_L",
    "INSOLE_VFO-NORM_R",
]

padding = dict(leftPadding=72, rightPadding=72, topPadding=72, bottomPadding=18)

portrait_frame = Frame(0, 0, *A4, **padding)
landscape_frame = Frame(0, 0, *landscape(A4), **padding)


def on_page(canvas, doc, pagesize=A4):

    page_num = canvas.getPageNumber()
    canvas.drawCentredString(pagesize[0] / 2, 50, str(page_num))

    # Draw header and footer in green background
    canvas.setFillColor("#1FAD77")
    canvas.rect(
        x=0,
        y=pagesize[1] - padding["topPadding"] / 1.1,
        width=pagesize[0],
        height=padding["topPadding"],
        fill=1,
        stroke=0,
    )
    canvas.rect(
        x=0,
        y=0,
        width=pagesize[0],
        height=padding["bottomPadding"] * 1.3,
        fill=1,
        stroke=0,
    )
    
    # Draw some info next to the logo
    canvas.setFillColor("white")
    canvas.setFont("Helvetica", 28)
    canvas.drawString(
        padding["leftPadding"] + 2.5 * inch,
        pagesize[1] - padding["topPadding"] * 3 / 5,
        "Biomarker Report",
    )

    # Draw the footer info
    canvas.setFont("Helvetica", 12)
    canvas.drawString(
        padding["leftPadding"],
        padding["bottomPadding"] / 2,
        "Report generated on {0}".format(datetime.now()),
    )

portrait_template = PageTemplate(
    id="portrait", frames=portrait_frame, onPage=on_page, pagesize=A4
)

def fig2image(f):
    buf = io.BytesIO()
    f.savefig(buf, format="png", dpi=300)
    buf.seek(0)
    x, y = f.get_size_inches()
    return Image(buf, x * inch, y * inch)


def df2table(df):
    table_data = []
    table_data.append(df.columns)
    for i, row in df.iterrows():
        table_data.append(list(row))

    table = Table(table_data)
    table_style = TableStyle(
        [
            ("BACKGROUND", (0, 0), (-1, 0), colors.grey),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.whitesmoke),
            ("ALIGN", (0, 0), (-1, -1), "CENTER"),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("BOTTOMPADDING", (0, 0), (-1, 0), 6),
            ("BACKGROUND", (0, 1), (-1, -1), "#EDF8F4"),
            ("TEXTCOLOR", (0, 1), (-1, -1), colors.black),
            ("ALIGN", (0, 1), (-1, -1), "CENTER"),
            ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
            ("INNERGRID", (0, 0), (-1, -1), 0.25, colors.grey),
            ("BOX", (0, 0), (-1, -1), 1, colors.grey),
            ("BOTTOMPADDING", (0, 1), (-1, -1), 8),
        ]
    )
    table.setStyle(table_style)
    return table


def create_demog(data, variables, cat_variables, path_field="PATHOLOGY"):

    demog = TableOne(
        data,
        columns=variables,
        categorical=cat_variables,
        groupby=path_field,
        pval=False,
        overall=False,
        missing=False,
    )
    demog_data = demog.tableone.reset_index()

    # remove columns and set new ones (if not, problems in df2table)
    unique_labels = data["PATHOLOGY"].unique().tolist()
    num_total_cols = len(demog_data.columns)
    num_unique_labels = len(unique_labels)
    num_blanks = num_total_cols - num_unique_labels
    blank_cols = [""] * num_blanks
    new_column_names = blank_cols + unique_labels
    demog_data.columns = new_column_names

    return demog_data


def wrap_text_in_cell(text, width=65):
    """
    Wraps the input text to a specified width using textwrap.fill.
    Handles non-string inputs by converting them to strings.
    """
    if text is None:
        return None
    try:
        text_str = str(text)
        return textwrap.fill(
            text_str, width=width, replace_whitespace=False, drop_whitespace=False
        )
    except Exception:
        return text


def create_doc(model_filename):
    doc = BaseDocTemplate("%s.pdf" % model_filename, pageTemplates=[portrait_template])
    return doc


def create_report(
    output_filename,
    data_input,
    path_field,
    severities,
    scatter_xvars,
    score_field="SCORES",
    retest_timelapse=200,
    output_dir=".",
):

    data = data_input.copy()
    doc = create_doc(output_filename)
    styles = getSampleStyleSheet()

    ## - demographics table and features used by the score
    demog = create_demog(
        data,
        list(["AGE", "PATIENT_SEX", "TEST_TYPE"] + severities + scatter_xvars),
        list(["TEST_TYPE", "PATIENT_SEX"] + severities),
        path_field,
    )

    features_used = [
        col for col in data.columns
        if col.startswith(("INSOLE", "TRUNK")) and col not in FEATURES_TO_REMOVE
    ]
    features_not_in_segments = [
        f for f in features_used if not f.startswith(tuple(FEATURE_FIELDS))
    ]
    if features_not_in_segments:
        print(f"Features used by the score but not listed in FEATURE_FIELDS: {features_not_in_segments}")

    features_table = pd.DataFrame(columns=["Segment", "List of features"])
    features_table["Segment"] = [
        f.replace("-NORM", "").replace("-MAX", "").replace("_", " ")
        for f in FEATURE_FIELDS
    ]
    list_of_features_used = list()
    for segment in FEATURE_FIELDS:
        features_segment = [
            f.split(segment)[1].replace("_", " ").replace("-", " ")
            for f in features_used
            if segment in f
        ]
        list_of_features_used.append(features_segment)
    features_table["List of features"] = list_of_features_used
    features_table["List of features"] = features_table["List of features"].apply(
        lambda x: ", ".join(x)
    )
    features_table["List of features"] = features_table["List of features"].apply(
        wrap_text_in_cell
    )
    features_table = features_table[features_table["List of features"] != ""]
    features_table = features_table.sort_values(by="Segment").reset_index(drop=True)

    ## - performance metrics
    roc_fig_nested, table_performance_nested = evaluate_model_performance(
        data, severities, score=score_field
    )
    bplot_fig = bplot_score(
        data, [path_field] + severities, score_field,
        os.path.join(output_dir, "boxplot_score.pdf"),
    )
    data.loc[data[path_field] == "Control", "OVERVIEW_DISTANCE-CALCULATED"] = np.nan
    data.loc[
        data["OVERVIEW_DISTANCE-CALCULATED"] < 25, "OVERVIEW_DISTANCE-CALCULATED"
    ] = np.nan
    scatter_fig = scatter_score(
        data, path_field, score_field, scatter_xvars, mode="scatter"
    )
    scatter_fig.savefig(
        os.path.join(output_dir, "scatter_score.png"), dpi=300, bbox_inches="tight"
    )

    ## - robustness metrics
    data_robust = data.drop_duplicates(subset=["ID_TEST"])
    test, retest = get_trt_data(
        data_robust[(data_robust["ELAPSED_DAYS"] < retest_timelapse)]
    )
    table_robustness = robustness_validation(test, retest, score_field)
    print(table_robustness)
    try:
        retest_fig = test_retest_plot(
            pd.concat([test, retest], axis=0),
            score_field,
            path_field,
            "ID_TEST",
            "RETEST",
        )
    except:
        retest_fig = plt.figure()

    ## - create story
    story = [
        Paragraph(f"Report generated for model {output_filename}", styles["Heading1"]),
        Paragraph("Generic demographics of the sample", styles["Heading2"]),
        df2table(demog),
        PageBreak(),
        Paragraph("Features used to compute the score", styles["Heading2"]),
        df2table(features_table),
    ]
    story.extend(
        [
            PageBreak(),
            Paragraph("Score evaluation", styles["Heading1"]),
            Paragraph("Performance metrics", styles["Heading2"]),
        ]
    )
    for i, roc_fig in enumerate(roc_fig_nested):
        story.extend(
            [
                Paragraph(severities[i], styles["Heading3"]),
                fig2image(roc_fig),
                df2table(table_performance_nested[i]),
            ]
        )
    story.extend(
        [
            PageBreak(),
            fig2image(bplot_fig),
            Spacer(1, 20),
            fig2image(scatter_fig),
            PageBreak(),
            Paragraph("Robustness metrics", styles["Heading2"]),
            df2table(table_robustness.round(4)),
            Spacer(1, 20),
            fig2image(retest_fig),
        ]
    )
    doc.build(story)

    return table_performance_nested, table_robustness

### 2.5 Longitudinal analysis
Changes in a clinical scale between visits are classified as *Improved / Stable / Worsened* using its MCID
(scales in `INVERT_SCALES` are those where a higher value means worse).
Score trajectories in each group are fitted with a linear mixed-effects model (random intercept per patient).

In [ ]:
INVERT_SCALES = [
    "SCALE_INCAT_total",
    "SCALE_INCAT_1",
    "SCALE_INCAT_2",
]


def classify_change(change, mcid, inv_scale=False):
    """Classify change based on MCID threshold.
    Parameters:
        change (float): The change value to classify.
        mcid (float): The minimal clinically important difference threshold.
        inv_scale (bool): If True, invert the classification (for scales where higher is worse
                          like INCAT or MICARS-SARA).
    Returns:
        str: "Improved", "Stable", or "Worsened" based on the change and MCID.
    """
    if inv_scale:
        change = -change
    if change >= mcid:
        return "Improved"
    elif -mcid < change < mcid:
        return "Stable"
    else:
        return "Worsened"


def get_clinical_worsening_data(
    data: pd.DataFrame,
    clinical_scale: str,
    mcid: float,
    score_name: str,
    key: str = "CODE",
    time_col: str = "ELAPSED_DAYS",
    id_col: str = "ID_TEST",
    mode: str = "consecutive",  # "consecutive" or "all_pairs"
) -> pd.DataFrame:
    # 1) Pre-filter, sort, and keep only repeated subjects
    df = (
        data.dropna(subset=[clinical_scale])
        .sort_values([key, time_col])
        .reset_index(drop=True)
    )

    repeated = df.groupby(key, sort=False).size()
    df = df[df[key].isin(repeated.index[repeated > 1])].reset_index(drop=True)

    inv_scale = clinical_scale in INVERT_SCALES

    out_frames = []

    # 2) Build pairs per subject group
    for patient_code, g in df.groupby(key, sort=False):
        n = len(g)
        if n < 2:
            continue

        if mode == "consecutive":
            g0 = g.iloc[:-1].copy()
            g1 = g.iloc[1:].copy()
        elif mode == "all_pairs":
            idx = g.index.to_numpy()
            pairs = np.fromiter(
                (i for pair in combinations(idx, 2) for i in pair), dtype=idx.dtype
            ).reshape(-1, 2)
            g0 = df.loc[pairs[:, 0]]
            g1 = df.loc[pairs[:, 1]]
        else:
            raise ValueError("mode must be 'consecutive' or 'all_pairs'")

        # 3) Vectorized differences
        scale_diff = g1[clinical_scale].to_numpy() - g0[clinical_scale].to_numpy()
        score_diff = g1[score_name].to_numpy() - g0[score_name].to_numpy()
        days_diff = g1[time_col].to_numpy() - g0[time_col].to_numpy()

        # 4) Marker classification
        marker = np.array(
            [
                np.nan if pd.isna(d) else classify_change(d, mcid, inv_scale)
                for d in scale_diff
            ]
        )

        # 5) Assemble one block per group
        out_frames.append(
            pd.DataFrame(
                {
                    key: patient_code,
                    "ID_TEST_initial": g0[id_col].to_numpy(),
                    "ID_TEST_final": g1[id_col].to_numpy(),
                    "clinical_scale_initial": g0[clinical_scale].to_numpy(),
                    "clinical_scale_final": g1[clinical_scale].to_numpy(),
                    "clinical_scale_diff": scale_diff,
                    "score_initial": g0[score_name].to_numpy(),
                    "score_final": g1[score_name].to_numpy(),
                    "score_diff": score_diff,
                    "days_difference": days_diff,
                    "marker": marker,
                }
            )
        )

    columns = [
        key, "ID_TEST_initial", "ID_TEST_final", "clinical_scale_initial",
        "clinical_scale_final", "clinical_scale_diff", "score_initial",
        "score_final", "score_diff", "days_difference", "marker",
    ]
    if out_frames:
        return pd.concat(out_frames, ignore_index=True)
    return pd.DataFrame(columns=columns)


def determine_trend(patient_data, clinical_scale, mcid):
    patient_data = patient_data.sort_values("ELAPSED_DAYS")

    first_value = patient_data.iloc[0][clinical_scale]
    last_value = patient_data.iloc[-1][clinical_scale]
    change = last_value - first_value
    return classify_change(change, mcid)


def fit_lmer(data, feature, scale, group=None):
    if group is None:
        plot_df = data.dropna(subset=[feature, scale], how="any").reset_index(drop=True)
        plot_df["Y"] = plot_df[feature]
        plot_df["X"] = plot_df[scale]
        md = smf.mixedlm("Y ~ X", plot_df, groups=plot_df["CODE"])
        mdf = md.fit()
        estimate = mdf.fe_params.iloc[1]
    else:
        plot_df = data.dropna(subset=[feature, scale, group], how="any").reset_index(drop=True)
        plot_df["Y"] = plot_df[feature]
        plot_df["X"] = plot_df[scale]
        plot_df["GROUP"] = plot_df[group]
        md = smf.mixedlm("Y ~ X * GROUP", plot_df, groups=plot_df["CODE"])
        mdf = md.fit()
        estimate = [mdf.fe_params.iloc[1], mdf.fe_params.iloc[3]]

    # predict fitted line
    try:
        plot_df["y_fit"] = mdf.predict(plot_df)
        r2_marginal, r2_conditional, icc = r2_computation(mdf, plot_df)
    except:
        plot_df["y_fit"] = np.nan
        r2_marginal = np.nan
        r2_conditional = np.nan
        icc = np.nan

    return plot_df, r2_marginal, r2_conditional, icc, estimate, mdf


def generate_lme_plots(ax, group_data, plot_data_sub, y_var, palette, group_label, result, estimate):
    # raw trajectories per patient
    sns.lineplot(
        data=group_data,
        x="ELAPSED_DAYS",
        y=y_var,
        hue="CODE",
        palette=palette,
        lw=1.5,
        ax=ax,
        errorbar=None,
    )
    ax.legend([], [], frameon=False)

    # fitted line from mixed model
    ax.plot(plot_data_sub["ELAPSED_DAYS"], plot_data_sub["y_fit"], "k-", linewidth=2)
    pval_lme = result.pvalues.get("X", np.nan)
    ax.set_title(f"{group_label} p={pval_lme:.4e}, fe={estimate*365:.2f}", fontsize=10)
    ax.set_ylabel(y_var)
    return ax

def r2_computation(mdf, data):
    var_residual = mdf.scale
    var_random = mdf.cov_re.values[0][0]
    var_fixed = mdf.predict(data).var()
    R2_m = var_fixed / (var_fixed + var_random + var_residual)
    R2_c = (var_fixed + var_random) / (var_fixed + var_random + var_residual)
    icc = var_random / (var_random + var_residual)
    return R2_m, R2_c, icc

## 3. Data loading and sample description
Number of patients and tests per group: pathology, balance severity (MICARS-SARA items 4 + 5), eyes-open and eyes-closed balancing, and ataxia (MICARS-SARA item 1).

In [ ]:
data_with_retrained_scores = pd.read_csv(SCORES_FILE)
data_with_retrained_scores = data_with_retrained_scores.loc[:, ~data_with_retrained_scores.columns.duplicated()]
print(f"Loaded {len(data_with_retrained_scores)} tests")

In [ ]:
for panel, col in [('PATHOLOGY', 'PATHOLOGY'),
                   ('MICARS_SARA_BALANCE_SEVERITY', 'MICARS_SARA_BALANCE_SEVERITY'),
                   ('A - OPEN_EYES_BALANCING', 'OPEN_EYES_BALANCING'),
                   ('B - CLOSE_EYES_BALANCING', 'CLOSE_EYES_BALANCING'),
                   ('C - ATAXIA', 'ATAXIA')]:
    print(f"\n── {panel} ──")
    summary = (
        data_with_retrained_scores.groupby(col)
        .agg(
            n_patients=('CODE', 'nunique'),
            n_tests=('ID_TEST', 'nunique')
        )
        .reset_index()
    )
    print(summary.to_string(index=False))
    print(f"  TOTAL: {summary['n_patients'].sum()} patients, {summary['n_tests'].sum()} tests")

## 4. Cross-sectional performance
Generates `SCORE_performance_report_test.pdf` with:
- demographics by group,
- ROC curves and performance tables (Control vs Pathology, Control vs Mild, Mild vs Moderate/Severe),
- score boxplots by pathology and balance severity,
- score vs MICARS-SARA scatter plots (Spearman),
- test-retest reliability (tests within `TIMELAPSE_RETEST` days).

In [ ]:
print("Evaluating cross-sectional performance...")
table_performance, table_robustness = create_report(
    os.path.join(OUTPUT_DIR, ""),
    data_with_retrained_scores,
    "PATHOLOGY",
    ["MICARS_SARA_BALANCE_SEVERITY"],
    SCALES_TO_COMPARE,
    score_field=SCORE_FIELD,
    retest_timelapse=TIMELAPSE_RETEST,
    output_dir=OUTPUT_DIR,
)
display(table_performance[0])

## 5. Score by clinical classification
Score distribution by pathology type and by severity of eyes-open balancing (MICARS-SARA item 4), eyes-closed balancing (item 5) and ataxia (item 1).
Significant pairwise differences (Mann-Whitney) are annotated.

In [ ]:
for group in ['PAT', 'OPEN_EYES_BALANCING', 'CLOSE_EYES_BALANCING', 'ATAXIA']:
    fig = bplot_score(
        data_with_retrained_scores,
        [group],
        SCORE_FIELD,
        os.path.join(OUTPUT_DIR, f"boxplot_score_{group}.png"),
    )
    plt.show()

## 6. Correlation with clinical scales

### 6.1 Merge clinical scales by session

In [ ]:
scales_db = pd.read_csv('smwt_db_mg.scales_quest.csv')
id_session_list = data_with_retrained_scores['ID_SESSION'].unique().tolist()
scales_db_filtered = scales_db[scales_db['ID_SESSION'].isin(id_session_list)]

scales_db_filtered = scales_db_filtered[['ID_SESSION',
    'SCALE_MICARS-SARA_total',
    'SCALE_MICARS-SARA_1',
    'SCALE_MICARS-SARA_4',
    'SCALE_MICARS-SARA_5',
    'SCALE_VIGORIMETER_1',
    'SCALE_VIGORIMETER_2',
    'SCALE_VIGORIMETER_total',
    'SCALE_MRC_total',
    'SCALE_MRC_total_L',
    'SCALE_MRC_total_R',
    'SCALE_INCAT_1',
    'SCALE_INCAT_2',
    'SCALE_INCAT_total',
    'QUESTIONNAIRE_EQ-5D_total',
    'QUESTIONNAIRE_PGIC_total']].copy()

data_with_retrained_scores_filtered = data_with_retrained_scores[[
    'ID_TEST', 'ID_SESSION', 'SCORE', 'CODE', 'PAT',
    'ELAPSED_DAYS', 'QUESTIONNAIRE_R-ODS_centile'
]].copy()
data_with_retrained_scores_filtered = data_with_retrained_scores_filtered.merge(
    scales_db_filtered, on='ID_SESSION', how='left'
)

### 6.2 Fill missing scales from the closest session
When a scale is missing in a session, it is taken from the closest session of the same patient within ±21 days.
Every imputed value is logged in `filling_log`.

In [ ]:
DAYS_BEFORE = 21
DAYS_AFTER = 21
EMPTY_VALUES = ['', ' ', 'nan', 'NaN', 'NAN', 'None', 'none', 'NA', 'N/A', '-', '--', 'null', 'NULL']


def normalize_sessions(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    df['ID_SESSION'] = df['ID_SESSION'].astype(str).str.strip()
    ext = df['ID_SESSION'].str.extract(r'^(.*)_(\d{4}-?\d{2}-?\d{2})')
    df['_PAT'] = ext[0].str.strip().str.lower()
    df['_DATE'] = pd.to_datetime(ext[1].str.replace('-', ''), format='%Y%m%d', errors='coerce')
    return df


data_norm = normalize_sessions(data_with_retrained_scores_filtered)
source = normalize_sessions(scales_db)

cols_id = ['ID_SESSION', 'CODE', '_PAT', '_DATE']
scale_cols = [c for c in source.columns if c not in cols_id and c in data_norm.columns]

for df in (data_norm, source):
    for c in scale_cols:
        df[c] = pd.to_numeric(df[c].replace(EMPTY_VALUES, np.nan), errors='coerce')

# ---------- Diagnostics ----------
print("Common scales:", len(scale_cols))
print("Unparsed dates → data:", data_norm['_DATE'].isna().sum(), "| scales_db:", source['_DATE'].isna().sum())
print("Patients in data:", data_norm['_PAT'].nunique(), "| in common with scales_db:",
      len(set(data_norm['_PAT']) & set(source['_PAT'])))

# ---------- Fill patient by patient ----------
data_with_retrained_scores_filled = data_norm.copy()
filling_records = []

for pat, rows_pat in data_norm.groupby('_PAT'):
    source_pat = source[source['_PAT'] == pat]

    for id_ses, rows_ses in rows_pat.groupby('ID_SESSION'):
        ses_date = rows_ses['_DATE'].iloc[0]
        missing_scales = [c for c in scale_cols if rows_ses[c].isna().all()]
        if not missing_scales:
            continue

        dif = (source_pat['_DATE'] - ses_date).dt.days
        nearby = source_pat.assign(DAYS_DIFF=dif, ABS_DAYS_DIFF=dif.abs())
        nearby = nearby[nearby['DAYS_DIFF'].between(-DAYS_BEFORE, DAYS_AFTER)] \
                       .sort_values(['ABS_DAYS_DIFF', 'DAYS_DIFF'])

        for col in missing_scales:
            with_value = nearby[nearby[col].notna()]
            if with_value.empty:
                continue
            origin = with_value.iloc[0]
            data_with_retrained_scores_filled.loc[rows_ses.index, col] = origin[col]
            filling_records.append({'PATIENT': pat, 'ID_SESSION': id_ses, 'SCALE': col,
                             'ID_SESSION_SOURCE': origin['ID_SESSION'],
                             'DAYS_DIFF': int(origin['DAYS_DIFF']), 'VALUE': origin[col]})

data_with_retrained_scores_filled = data_with_retrained_scores_filled.drop(columns=['_PAT', '_DATE'])
filling_log = pd.DataFrame(filling_records, columns=['PATIENT', 'ID_SESSION', 'SCALE',
                                                   'ID_SESSION_SOURCE', 'DAYS_DIFF', 'VALUE'])
filling_log.to_csv(os.path.join(OUTPUT_DIR, ""), index=False)

before = data_norm[scale_cols].isna().sum()
after = data_with_retrained_scores_filled[scale_cols].isna().sum()
display(pd.DataFrame({'NaN_BEFORE': before, 'FILLED': before - after, 'NaN_AFTER': after}))

### 6.3 Reference values for healthy controls
Controls are not assessed with the clinical scales. They are assigned reference values:
-1 for MICARS-SARA, INCAT, EQ-5D and PGIC; maximum MRC (60, 30 per side); R-ODS centile 100.

In [ ]:
mask = data_with_retrained_scores_filled['PAT'] == ID_CONTROL

cols_minus1 = [
    'SCALE_MICARS-SARA_total',
    'SCALE_MICARS-SARA_1',
    'SCALE_MICARS-SARA_4',
    'SCALE_MICARS-SARA_5',
    'SCALE_INCAT_1',
    'SCALE_INCAT_2',
    'SCALE_INCAT_total',
    'QUESTIONNAIRE_EQ-5D_total',
    'QUESTIONNAIRE_PGIC_total'
]
for col in cols_minus1:
    data_with_retrained_scores_filled.loc[mask, col] = -1

data_with_retrained_scores_filled.loc[mask, 'SCALE_MRC_total']   = 60
data_with_retrained_scores_filled.loc[mask, 'SCALE_MRC_total_L'] = 30
data_with_retrained_scores_filled.loc[mask, 'SCALE_MRC_total_R'] = 30

data_with_retrained_scores_filled.loc[mask, 'QUESTIONNAIRE_R-ODS_centile'] = 100

### 6.4 Spearman correlation matrix
Pairwise deletion of missing values.

In [ ]:
cols_corr = [
    'SCORE',
    'SCALE_MICARS-SARA_total',
    'SCALE_MICARS-SARA_1',
    'SCALE_MICARS-SARA_4',
    'SCALE_MICARS-SARA_5',
    'QUESTIONNAIRE_R-ODS_centile',
    'SCALE_MRC_total',
    'SCALE_MRC_total_L',
    'SCALE_MRC_total_R',
    'SCALE_INCAT_1',
    'SCALE_INCAT_2',
    'SCALE_INCAT_total',
    'QUESTIONNAIRE_EQ-5D_total',
    'QUESTIONNAIRE_PGIC_total'
]
cols_corr = [c for c in cols_corr if c in data_with_retrained_scores_filled.columns]
df_corr = data_with_retrained_scores_filled[cols_corr].copy()

corr_matrix = df_corr.corr(method='spearman')

corr_matrix = corr_matrix.rename(
    index={
        'SCALE_MICARS-SARA_4': 'SCALE_MICARS-SARA_5',
        'SCALE_MICARS-SARA_5': 'SCALE_MICARS-SARA_6'
    },
    columns={
        'SCALE_MICARS-SARA_4': 'SCALE_MICARS-SARA_5',
        'SCALE_MICARS-SARA_5': 'SCALE_MICARS-SARA_6'
    }
)

plt.figure(figsize=(14, 12))
mask_tri = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask_tri,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title('Correlation Matrix (Spearman)', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, ""), dpi=300, bbox_inches="tight")
plt.show()

corr_matrix.to_csv(os.path.join(OUTPUT_DIR, ""))
corr_matrix['SCORE'].sort_values(ascending=False)

## 7. Longitudinal analysis anchored on R-ODS
Patients only. The score is expressed ×100.

1. All pairs of visits of each patient are classified with the R-ODS centile MCID (8 points).
2. Each patient is assigned a trend (first vs last visit), and R-ODS and score trajectories are fitted with a linear mixed-effects model per group.

In [ ]:
data_with_retrained_scores_filled['SCORE'] = data_with_retrained_scores_filled['SCORE'] * 100
data_with_retrained_scores_filled['PATHOLOGY'] = data_with_retrained_scores_filled['PAT'].apply(
    lambda x: 'Control' if x == ID_CONTROL else 'Pathology'
)
df_merged = data_with_retrained_scores_filled.copy()

In [ ]:
clinical_scale = 'QUESTIONNAIRE_R-ODS_centile'
mcid = 8
score_name = 'SCORE'

data_validation = df_merged[df_merged['PAT'] != ID_CONTROL].copy()
data_validation = data_validation.loc[:, ~data_validation.columns.duplicated()]

print(f"Analyzing patients with clinical change using '{clinical_scale}' (MCID = {mcid})...")
long_data = get_clinical_worsening_data(
    data_validation,
    clinical_scale,
    mcid,
    score_name,
    mode="all_pairs",
)
long_data.to_csv(os.path.join(OUTPUT_DIR, f"longitudinal_data_{clinical_scale}_{mcid}.csv"), index=False)

for marker in ["Stable", "Worsened", "Improved"]:
    sub = long_data[long_data["marker"] == marker]
    print(
        f"\n  {marker}: {len(sub)}/{len(long_data)} ({len(sub)/len(long_data)*100:.1f}%)"
        f"\n  • Mean {clinical_scale} change: {sub['clinical_scale_diff'].mean():.2f} ± {sub['clinical_scale_diff'].std():.2f}"
        f"\n  • Mean {score_name} change: {sub['score_diff'].mean():.2f} ± {sub['score_diff'].std():.2f}"
    )

In [ ]:
plot_df_4 = (
    data_validation
    .dropna(subset=[clinical_scale, "ELAPSED_DAYS"], how="any")
    .reset_index(drop=True)
)

## - trend per patient (first vs last visit)
trend_results_4 = plot_df_4.groupby(
    "CODE", group_keys=False
).apply(determine_trend, clinical_scale=clinical_scale, mcid=mcid)
trend_results_4_df = trend_results_4.reset_index(name="Trend")
plot_df_4 = plot_df_4.merge(trend_results_4_df, on="CODE", how="left")
print(trend_results_4_df["Trend"].value_counts())

## - LME per group
fig, axes = plt.subplots(3, 2, figsize=(10, 10), sharex=True, sharey="col")
groups = ["Worsened", "Stable", "Improved"]
palettes = ["Blues", "Greens", "Oranges"]

for i, (group, palette) in enumerate(zip(groups, palettes)):
    group_data = plot_df_4[plot_df_4["Trend"] == group]
    print(f"\n  {group} (n tests = {len(group_data)}):")

    for j, y_var in enumerate([clinical_scale, score_name]):
        plot_data_sub, r2_marginal, r2_conditional, icc, estimate, result = (
            fit_lmer(group_data, y_var, "ELAPSED_DAYS")
        )
        print(
            f"  • {y_var} - Fixed-effects trend: {estimate*365:.2f} points/year; p-value = {result.pvalues['X']:.4e}"
            f"    ICC = {icc:.2f}, R²c = {r2_conditional:.2f}; R²m = {r2_marginal:.2f}"
        )
        generate_lme_plots(
            ax=axes[i, j],
            group_data=group_data,
            plot_data_sub=plot_data_sub,
            y_var=y_var,
            palette=palette,
            group_label=group,
            result=result,
            estimate=estimate,
        )
print("    *R²m includes fixed effects (elapsed time) whereas R²c includes the whole model. "
      "Higher ICC indicates higher reliability/consistency of the measure (or less dispersion/noise)")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, f"lme_trends_{clinical_scale}_{mcid}.pdf"), dpi=200, bbox_inches="tight")
plt.show()

## 8. Score change vs MICARS-SARA (Figure 5)
For every pair of visits of the same patient, a score change between 25% and 100% is classified as *improvement* (score ↑) or *worsening* (score ↓).
The figure shows the MICARS-SARA total at the start and end of each pair, with the group mean and its 95% CI.

In [ ]:
clinical_columns = [
    'QUESTIONNAIRE_R-ODS_centile',
    'QUESTIONNAIRE_EQ-5D_total',
    'SCALE_MICARS-SARA_total',
    'SCALE_MRC_total',
    'SCALE_VIGORIMETER_total',
    'SCALE_INCAT_total',
]
clinical_col = 'SCALE_MICARS-SARA_total'

df_valid_tests = df_merged.copy()
big_change_records = []

# For each patient, evaluate all pairs of visits (t0->t1, t0->t2, t1->t2, etc.)
for pid, group in df_valid_tests.groupby('CODE'):
    visits = group[['ELAPSED_DAYS', 'SCORE'] + clinical_columns].reset_index(drop=True)
    n = len(visits)

    for i in range(n - 1):
        for j in range(i + 1, n):
            v0 = visits.loc[i]
            v1 = visits.loc[j]

            if v0['SCORE'] == 0:
                continue

            pct_change = (v1['SCORE'] - v0['SCORE']) / v0['SCORE'] * 100

            if 25 <= abs(pct_change) <= 100:
                big_change_records.append({
                    'CODE': pid,
                    'ELAPSED_DAYS_start': v0['ELAPSED_DAYS'],
                    'ELAPSED_DAYS_end': v1['ELAPSED_DAYS'],
                    'SCORE_start': v0['SCORE'],
                    'SCORE_end': v1['SCORE'],
                    'SCORE_change_pct': pct_change,
                    'CHANGE_DIRECTION': 'improve' if v1['SCORE'] > v0['SCORE'] else 'worsen',
                    f'{clinical_col}_start': v0[clinical_col],
                    f'{clinical_col}_end': v1[clinical_col]
                })

df_big_change = pd.DataFrame(big_change_records)
df_big_change[f'{clinical_col}_diff'] = df_big_change[f'{clinical_col}_end'] - df_big_change[f'{clinical_col}_start']
df_big_change[f'{clinical_col}_pct_change'] = df_big_change[f'{clinical_col}_diff'] / df_big_change[f'{clinical_col}_start'] * 100
df_big_change.to_csv(os.path.join(OUTPUT_DIR, f"score_change_pairs_{clinical_col}.csv"), index=False)

df_improve = df_big_change[df_big_change['CHANGE_DIRECTION'] == 'improve']
df_worsen = df_big_change[df_big_change['CHANGE_DIRECTION'] == 'worsen']
print(f"Pairs with score worsening: {len(df_worsen)} | improvement: {len(df_improve)}")

In [ ]:
def ci95(x):
    if len(x) > 1:
        return stats.t.interval(0.95, len(x) - 1, loc=np.mean(x), scale=stats.sem(x))
    return (np.nan, np.nan)


fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for i, (df_subset, title, color) in enumerate(zip(
    [df_worsen, df_improve],
    ['Worsening (Score ↓ ≥25%)', 'Improvement (Score ↑ ≥25%)'],
    ['#E69F00', '#0072B2']
)):
    ax = axes[i]

    for _, row in df_subset.iterrows():
        ax.plot([0, 1],
                [row[f'{clinical_col}_start'], row[f'{clinical_col}_end']],
                color='lightgray', alpha=0.3, linewidth=1)

    start_vals = df_subset[f'{clinical_col}_start'].dropna()
    end_vals = df_subset[f'{clinical_col}_end'].dropna()
    mean_start = start_vals.mean()
    mean_end = end_vals.mean()

    ax.plot([0, 1], [mean_start, mean_end], linewidth=4, color=color, label='Mean')

    ci_start = ci95(start_vals)
    ci_end = ci95(end_vals)
    ax.fill_between([0, 1], [ci_start[0], ci_end[0]], [ci_start[1], ci_end[1]], color=color, alpha=0.15)

    pct_change = ((mean_end - mean_start) / mean_start) * 100 if mean_start != 0 else np.nan
    point_change = mean_end - mean_start
    sign = "+" if point_change > 0 else ""

    ax.text(0.5, max(df_subset[f'{clinical_col}_end'].max(), df_subset[f'{clinical_col}_start'].max()) * 0.9,
            f'Δ {sign}{point_change:.2f} points\n({pct_change:.1f}%)',
            ha='center', fontsize=12, fontweight='bold')

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Start', 'End'])
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_ylabel(clinical_col)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, ""), dpi=300, bbox_inches='tight')
plt.show()